# **Introduction to Fourier transform in two dimensions**
## **Author: Dr. Rahul Remanan**

### $\text{First published: April } 3^{rd} \text{, 2026}$

### $\text{Last updated: October } 10^{th} \text{, 2026}$

In [ ]:
import os, math, time, random, numpy as np, scipy as sp, \
       matplotlib.image as mpimg, matplotlib.pyplot as plt

from scipy.stats import norm
from functools import cache, reduce

# Handling complex numbers:

Every non-zero complex number ${z = x+ iy}$ can be written in the form: ${re}^{iθ}$, where the value: ${r = |z| = {\sqrt{{x^{2}+y^{2}}}}}$ is referred to as the **magnitude** of $z$, and $θ$ as the **phase, angle, or argument** of $z$.

## Getting the magnitude of a complex number array:

In [ ]:
print(

    '\n',

    'Magnitude: ', np.abs([1.0, 1.0j, 1+1j, -1-1j])

)

## Getting the phase of a complex number array:

In [ ]:
print(

    '\n',

    'In radians: ', np.angle([1.0, 1.0j, 1+1j, -1-1j]),

    '\n',

    'In degrees: ', np.angle([1.0, 1.0j, 1+1j, -1-1j], deg=True),

    '\n',

    np.angle([0., -0., complex(0., -0.), complex(-0., -0.)])   # convention

)

## Getting the sign of a complex number array:

The [sign function](https://numpy.org/doc/stable/reference/generated/numpy.sign.html) is defined as: $S(x + iy) = \frac{(x + iy)}{\sqrt{x^2+y^2}}$

In [ ]:
np.sign(

    [1.0, 1.0j, 1 + 1j, -1 - 1j]

)

## Separating real and imaginary parts of a complex number array:

$\text{For } c \in ℂ \text{ , where: } c = a + ib \text{; the real part is: } a \text{ and the imaginary part is: } b.$

In [ ]:
c = random.SystemRandom().random() + (random.SystemRandom().random()* 1j)

print(

    f'For the complex number: {c} the real component is: {c.real} and the imaginary component is: {c.imag} ...'

)

# Discrete Fourier transform (DFT) and inverse discrete Fourier transform (iDFT):

In [ ]:
a = np.random.uniform(0, 1, (4, 6))

print(a)

In [ ]:
N = np.prod(a.shape)

## **Discrete Fourier transform:**

Discrete Fourier transform of a two dimensional $M{\times}N$ matrix, decomposes each value of the matrix: $r_{n}{\text{, where: }}r{\in}{ℝ}$,  into the form: ${c_{n}} = {a_{n}}({cos}({\theta}_{n}) + {i}{\cdot}{sin}({\theta}_{n}))$, through the application of: 
* [Euler's formula](https://en.wikipedia.org/wiki/Euler's_formula): ${e^{i{\cdot}{x}}={{\cos{x}}+{i}{\cdot}{\sin{x}}}}$
* [Improper Reimann integral](https://en.wikipedia.org/wiki/Improper_integral).

${F(a,b)={{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}f(k,l){e^{{-i}{2}{\pi}{({\frac{ka}{N}}+{\frac{lb}{M}})}}}}$

In [ ]:
@cache

def fourier_transform_exp(

    d:'Dictionary key lookup vector',

    dtype:str='complex64'

)->'Vector of a specified dtype, depending on the keyword argument: dtype':

    e_ft = np.exp(-1 * (np.array([0 + 1j]) * 2 * math.pi * d))

    return e_ft.astype(getattr(np, dtype))

In [ ]:
def compute_summation_array(

    inp_arr:np.ndarray,

    dtype:str='complex64'

)->list:

    inp_arr_list = []

    for i in range(inp_arr.shape[0]):

        for j in range(inp_arr.shape[1]):

            try:

                inp_val_shape = inp_arr[i, j].shape[0]

                assert inp_val_shape == 1, f'Received invalid shape: {inp_val_shape} ...'

                inp_arr_list.append(

                    np.array(

                        [

                            inp_arr[i, j][0]

                        ]

                    ).astype(

                        getattr(

                            np, 

                            dtype

                        )

                    )

                )

            except:

                inp_arr_list.append(

                    np.array(

                        [

                            inp_arr[i, j]

                        ]

                    ).astype(

                        getattr(

                            np, 

                            dtype

                        )

                    )

                )

    return inp_arr_list

In [ ]:
def lookup_and_update_transform_function_dict(

    d:'Dictionary key lookup vector',

    transform_dict:dict,

    transform_fn:callable,

    dtype:str='complex64',

    display_errors:bool=False,

    verbose:bool=False

)->tuple:

    try:

        t_val = transform_dict[d]

    except Exception as e:

        transform_dict.update(

            {

                d : transform_fn(d, dtype)

            }

        )

        if display_errors and verbose:

            print(

                f'Failure updating look up dictionary that already has {len(transform_dict)} items due to exception: {e} ...',

                end = '\r',

                flush=True

            )

        t_val = transform_dict[d]

    return t_val, transform_dict

In [ ]:
def compute_transformed_summation_matrix(

    inp_arr_list:list,

    transform_dict:dict,

    array_index_i:int,

    array_index_j:int,

    inp_height:int,

    inp_width:int,

    tolerance:int=8,

    transform_fn:callable=None,

    dtype:str='complex64',

    return_sum:bool=False,

    heuristics:bool=False,

    display_errors:bool=False,

    verbose:bool=False

)->tuple:

    idx = 0

    t_sum = []

    if return_sum:

        t_sum = 0

        t_sum = np.array([t_sum]).astype(getattr(np, dtype))

    if heuristics:

        t_k = []

    for k in range(inp_height):

        if heuristics:

            t_l = []

        for l in range(inp_width):

            d1 = k * array_index_i / inp_height

            d1 = d1 - math.floor(d1)

            d2 = l * array_index_j / inp_width

            d2 = d2 - math.floor(d2)

            t_val = inp_arr_list[idx].astype(getattr(np, dtype))

            idx += 1

            if np.isclose(d1 + d2, 0.0, rtol=tolerance):

                if return_sum:

                    t_sum += t_val

                else:

                    t_sum.append(t_val)

            else:

                t_val_1, transform_dict = lookup_and_update_transform_function_dict(

                    d1,

                    transform_dict,

                    transform_fn,

                    dtype,

                    display_errors,

                    verbose
                )

                t_val_2, transform_dict = lookup_and_update_transform_function_dict(

                    d2,

                    transform_dict,

                    transform_fn,

                    dtype,

                    display_errors,

                    verbose

                )

                t_sum += t_val * t_val_1 * t_val_2

            if heuristics:

                t_l.append(t_val)

        if heuristics:

            t_k.append(t_l)

    if heuristics:

        return t_sum, transform_dict, t_k

    return t_sum, transform_dict

In [ ]:
def compute_two_dimensional_transform(

    inp_arr:np.ndarray,

    tolerance:int=8,

    dtype:str='complex64',

    transform_fn:callable=None,

    heuristics:bool=False,

    display_errors:bool=False,

    verbose:bool=False

)->np.ndarray:

    t_start_time = time.time()

    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'

    assert transform_fn is not None, 'Expected an input in transform_fn keyword argument, but received: None ...'

    assert callable(transform_fn), f'Expected a function to be passed using the keyword argument: transform_fn, instead received an instance of: {type(transform_fn)} ...'

    N, M = inp_arr.shape[0], inp_arr.shape[1]

    arr_t = np.zeros_like(inp_arr).astype(getattr(np, dtype))

    prog_idx = 0

    fourier_dict = dict()

    inp_arr_list = compute_summation_array(inp_arr, dtype)

    return_sum = True

    if heuristics:

        ti_val = []

    t_sum_list = []

    for i in range(arr_t.shape[0]):

        if heuristics:

            tj_val = []

        for j in range(arr_t.shape[1]):

            start_time = time.time()

            prog_idx += 1

            
            s_mat_out = compute_transformed_summation_matrix(

                inp_arr_list,

                fourier_dict,

                i, j, N, M,

                tolerance,

                transform_fn,

                dtype,

                return_sum,

                heuristics,

                display_errors,

                verbose

            )

            if heuristics:

                t_sum, fourier_dict, t_k = s_mat_out[0], s_mat_out[1], s_mat_out[2]

            else:

                t_sum, fourier_dict = s_mat_out[0], s_mat_out[1]

            del s_mat_out

            arr_t[i, j] += t_sum[0]
            
            prog_frac = prog_idx / (N * M)

            end_time = time.time()

            if verbose:

                print(

                    ' Completed: ', prog_frac, ' / 1.0,',

                    ' Time remaining: ', int(np.round((end_time - start_time) * (1 - prog_frac) * N * M)), ' (s),',

                    ' Time elapsed: ', int(np.round(end_time - t_start_time)), ' (s) ...',

                    ' '*100,

                    end = '\r',

                    flush=True
                )

            if heuristics:

                t_sum_list.append(t_sum)

                tj_val.append(t_k)

            start_time = time.time()

        if heuristics:

            ti_val.append(tj_val)

    t_end_time = time.time()

    if verbose:

        exe_time = int(np.round(t_end_time - t_start_time))

        if exe_time == 0:

            exe_time = t_end_time - t_start_time

        print(

            'Two-dimensional transform computed in: ', exe_time, ' (s) ...', ' '*100,

            end = '\r',
    
            flush=True

        )

    if heuristics:

        return np.array(arr_t).reshape(N, M), t_sum_list, ti_val

    return np.array(arr_t).reshape(N, M)

In [ ]:
def compute_fourier_transform(

    inp_arr:np.ndarray,

    tolerance:int=8,

    dtype:str='complex64',

    transform_fn:callable=None,

    heuristics:bool=False,

    display_errors:bool=False,

    verbose:bool=False

)->'Python tuple or a NumPy ndarray, depending on the keyword argument: heuristics':

    return compute_two_dimensional_transform(

               inp_arr,

               tolerance,

               dtype,

               fourier_transform_exp,

               heuristics,

               display_errors,

               verbose

            )

In [ ]:
a_ft, _, fi_val = compute_fourier_transform(

    a, heuristics=True, verbose=True

)

In [ ]:
print(

    f'\nFourier transform output: \n{a_ft}'

)

## **Discrete two dimensional fast Fourier transform:**

The discrete fast Fourier transform is a computational time saving strategy by using the computational memory to store previously computed Fourier transform matrix components. 

This implementation of the fast Fourier transform preserves the dimensionality of the input array.

In [ ]:
@cache

def theta_in_fractions_to_radians(theta:'Vector')->'Vector':

    return 2 * math.pi * theta

In [ ]:
def compute_fourier_coefficients_matrix(

    cos_coef_mat:list,

    sin_coef_mat:list

)->np.ndarray:

    return np.array(cos_coef_mat) + (np.array(sin_coef_mat)* 1j)

In [ ]:
def compute_row_column_fourier_coefficients(

    num_rows:int,

    num_cols:int,

    tolerance:int=8

)->tuple:

    cos_col_coef_mat, sin_col_coef_mat = [], []

    cos_row_coef_mat, sin_row_coef_mat = [], []

    col_theta, row_theta = 1 / num_cols, 1 / num_rows

    for _ in range(num_rows):

        for m in range(num_cols):
    
            m_theta = (1 - (m * col_theta))

            cos_col_coef_mat.append(

                round(

                    math.cos(

                        theta_in_fractions_to_radians(

                            m_theta

                        )

                    ), tolerance

                )

            )
        
            sin_col_coef_mat.append(

                round(

                    math.sin(

                        theta_in_fractions_to_radians(

                            m_theta

                        )

                    ), tolerance

                )
                
            )

    for n in range(num_rows):
    
        for _ in range(num_cols):
    
            n_theta = 1 - (n * row_theta)
    
            cos_row_coef_mat.append(

                round(

                    math.cos(

                        theta_in_fractions_to_radians(

                            n_theta

                        )

                    ), tolerance

                )

            )
    
            sin_row_coef_mat.append(

                round(

                    math.sin(

                        math.pi - theta_in_fractions_to_radians(

                            n_theta

                        )

                    ), tolerance

                )

            )

    return cos_col_coef_mat, sin_col_coef_mat, cos_row_coef_mat, sin_row_coef_mat

In [ ]:
def compute_fast_two_dimensional_transform(

    inp_arr:np.ndarray,

    tolerance:int=8,

    update_threshold:float=0.01,

    transform_fn:callable=None,

    transform_coefficients_matrix_fn:callable=None,

    dtype:str='complex64',

    use_symmetry:bool=False,

    reshape_to_square_matrix:bool=False,

    display_errors:bool=False,

    heuristics:bool=False,

    verbose:bool=False

)->np.ndarray:

    assert transform_fn is not None, 'Expected a valid input for the keyword argument: transform_fn, instead received None ...'

    assert transform_coefficients_matrix_fn is not None, 'Expected a valid input for the keyword argument: transform_coefficients_matrix_fn, instead received None ...'

    assert callable(transform_fn), f'Expected a callable object or a function as the input for the keyword argument: transform_fn, instead received an instance of: {type(transform_fn)} ...'

    assert callable(transform_coefficients_matrix_fn), f'Expected a callable object or a function as the input for the keyword argument: transform_coefficients_matrix_fn, instead received an instance of: {type(transform_coefficients_matrix_fn)} ...'

    if verbose:

        t_start_time = time.time()

    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'

    N, M = inp_arr.shape

    if use_symmetry:

        if N % 2 != 0:

            inp_arr = np.insert(inp_arr, 0, 0, axis=0)

        if M % 2 != 0:

            inp_arr = np.insert(inp_arr, 0, 0, axis=1)

        N, M = inp_arr.shape

    if reshape_to_square_matrix:

        s_delta = N - M

        if s_delta > 0:

            for pad in range(s_delta):

                inp_arr = np.insert(inp_arr, 0, 0, axis=1)

        elif s_delta < 0:

            for pad in range(-s_delta):

                inp_arr = np.insert(inp_arr, 0, 0, axis=0)

        N, M = inp_arr.shape

        assert N == M, f'Reshaping the input matrix into a square matrix of {N} x {M} dimensions failed ...'

    a_t = np.zeros_like(inp_arr).astype(getattr(np, dtype))

    prog_idx = 0

    transform_dict = dict()

    i, j = 0, 0

    if verbose:

        start_time = time.time()

    inp_arr_list = compute_summation_array(inp_arr, dtype)

    ts_mat_out = compute_transformed_summation_matrix(

        inp_arr_list,

        transform_dict,

        i, j, N, M,

        tolerance,

        transform_fn,

        dtype,

        heuristics=heuristics,

        verbose=verbose

    )

    del inp_arr_list

    if heuristics:

        t_val_list, transform_dict, t_k = ts_mat_out[0], ts_mat_out[1], ts_mat_out[2]

    else:

        t_val_list, transform_dict = ts_mat_out[0], ts_mat_out[1]

    del ts_mat_out

    if verbose:

        print(

            'Computed the summation matrix for the fast two dimensional transform, of length: ', 

            len(t_val_list)

        )

    t_val_arr = np.array(t_val_list)

    cos_col_coef_mat, \
    sin_col_coef_mat, \
    cos_row_coef_mat, \
    sin_row_coef_mat = compute_row_column_fourier_coefficients(

        N, M, tolerance

    )

    row_coef_mat = np.ones((N, M)).astype(getattr(np, dtype))

    col_coef_mat = np.ones_like(row_coef_mat).astype(getattr(np, dtype))

    t_sum_mat    = np.ones_like(row_coef_mat).astype(getattr(np, dtype))

    cache_tolerance = tolerance - 2

    assert cache_tolerance > 2, f'Supplied tolerance value of {tolerance} for computing fast fourier transform is too low ...'

    a_t = np.zeros((N, M)).astype(getattr(np, dtype))

    for n in range(N):

        if n != 0:

            row_coef_mat *= np.round(

                transform_coefficients_matrix_fn(

                    [cos_row_coef_mat],

                    [sin_row_coef_mat]

                ).reshape(row_coef_mat.shape),

                cache_tolerance

            )

        for m in range(M):

            if m > 0:

                col_coef_mat *= np.round(
                    transform_coefficients_matrix_fn(

                        [cos_col_coef_mat],

                        [sin_col_coef_mat]

                    ).reshape(col_coef_mat.shape), 
                    
                    cache_tolerance

                )

            else:

                col_coef_mat = np.ones_like(row_coef_mat).astype(getattr(np, dtype))

            t_sum_mat *= np.round(col_coef_mat, cache_tolerance)

            t_sum_mat *= np.round(row_coef_mat, cache_tolerance)

            t_sum_mat *= np.round(t_val_arr.reshape(N, M), cache_tolerance)

            t_sum = np.sum(t_sum_mat)

            del t_sum_mat

            t_sum_mat = np.ones_like(row_coef_mat).astype(getattr(np, dtype))

            a_t[n][m] = t_sum 

            del t_sum

            if verbose:

                prog_idx += 1

                prog_frac = prog_idx / (N * M)

                end_time = time.time()

                if prog_frac >= update_threshold:

                    print(

                        ' Completed: ', np.round(prog_frac, 2), ' / 1.0,',

                        ' Time remaining: ', int(np.round((end_time - start_time) * (1 - prog_frac) * N * M)), ' (s),',

                        ' Time elapsed: ', int(np.round(end_time - t_start_time)), ' (s) ...',

                        ' '*75,

                        end = '\r',

                        flush=True

                    )

                    update_threshold += 0.01

            if verbose:

                start_time = time.time()

    del cos_col_coef_mat, sin_col_coef_mat

    del cos_row_coef_mat, sin_row_coef_mat

    if verbose:

        t_end_time = time.time()

        exe_time = t_end_time - t_start_time

        if int(np.round(t_end_time - t_start_time)) != 0:

            exe_time = int(np.round(t_end_time - t_start_time))

        print(

            'Two dimensional fast transform computed in: ', exe_time, ' (s) ...', ' '*50,

            end = '\r',

            flush=True

        )

    return np.array(a_t).reshape(N, M)

In [ ]:
def compute_fast_fourier_transform(

    inp_arr:np.ndarray,

    tolerance:int=8,

    update_threshold:float=0.01,

    dtype:str='complex64',

    use_symmetry:bool=False,

    reshape_to_square_matrix:bool=False,

    display_errors:bool=False,

    heuristics:bool=False,

    verbose:bool=False

)->np.ndarray:

    return compute_fast_two_dimensional_transform(

        inp_arr,

        tolerance,

        update_threshold,

        fourier_transform_exp,

        compute_fourier_coefficients_matrix,

        dtype,

        use_symmetry,

        reshape_to_square_matrix,

        display_errors,

        heuristics,

        verbose

    )

In [ ]:
a_fft = compute_fast_fourier_transform(
    
    a, verbose=True

)

In [ ]:
print(
    
    f'\nFourier transform output: \n{a_fft}'

)

### Compare outputs between the fast Fourier transform and regular Fourier transform functions:

In [ ]:
assert(

    np.isclose(

        np.array(a_fft).flatten(),

        np.array(a_ft).flatten(),

        rtol=6

    )

).all()

## **Inverse discrete Fourier transform:**

Inverse discrete Fourier transform of a two dimensional $M{\times}N$ matrix, recomposes each value of the matrix: ${{c_{n}}{\text{, where: c}}{\in}{ℂ}}$, into the corresponding value: ${r_{n}}{\text{, where: r}}{{\in}ℝ}$, of another $M{\times}N$ matrix; which was used to derive the discrete Fourier transformed complex matrix. 

The inverse discrete Fourier transform function maps: ${c_{n}{\rightarrow}r_{n}}$, through the application of:
* [Euler's formula](https://en.wikipedia.org/wiki/Euler's_formula): ${e^{i{\cdot}{x}}={{\cos{x}}+{i}{\cdot}{\sin{x}}}}$
* [Improper Reimann integral](https://en.wikipedia.org/wiki/Improper_integral).

${f(x,y)={{\frac{1}{N{\cdot}M}}{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}F(k,l){e^{{i}{2}{\pi}{({\frac{kx}{N}}+{\frac{ly}{M}})}}}}$

In [ ]:
@cache

def inverse_fourier_transform_exp(

    d:'Dictionary key lookup vector',

    dtype='complex64'

)->'Vector of a specified dtype, depending on the keyword argument: dtype':

    e_ift = np.exp(np.array([0 + 1j]) * 2 * math.pi * d)

    return e_ift.astype(getattr(np, dtype))

In [ ]:
def compute_inverse_fourier_transform(

    inp_arr:np.ndarray,

    tolerance:int=8,

    transform_fn:callable=None,

    dtype:str='complex64',

    heuristics:bool=False,

    display_errors:bool=False,

    verbose:bool=False

)->'Python tuple or a NumPy ndarray, depending on the keyword argument: heuristics':

    out_ift = compute_two_dimensional_transform(

        inp_arr,

        tolerance,

        dtype,

        inverse_fourier_transform_exp,

        heuristics,

        display_errors,

        verbose

    )

    if heuristics:

        return out_ift[0] / np.prod(inp_arr.shape), out_ift[1], out_ift[2]

    return out_ift / np.prod(inp_arr.shape)

In [ ]:
def compute_inverse_fourier_coefficients_matrix(

    cos_coef_mat:list,

    sin_coef_mat:list

)->np.ndarray:

    return np.array(cos_coef_mat) - (np.array(sin_coef_mat)) * 1j

In [ ]:
def compute_inverse_fast_fourier_transform(

    inp_arr:np.ndarray,

    tolerance:int=8,

    update_threshold:float=0.01,

    dtype:str='complex64',

    use_symmetry:bool=False,

    reshape_to_square_matrix:bool=False,

    display_errors:bool=False,

    heuristics:bool=False,

    verbose:bool=False

)->np.ndarray:

    return compute_fast_two_dimensional_transform(

        inp_arr,

        tolerance,

        update_threshold,

        inverse_fourier_transform_exp,

        compute_inverse_fourier_coefficients_matrix,

        dtype,

        use_symmetry,

        reshape_to_square_matrix,

        display_errors,

        heuristics,

        verbose

    ) / np.prod(np.shape(inp_arr))

In [ ]:
a_ift, ift_sum_list, ifti_val = compute_inverse_fourier_transform(

    a_ft,

    heuristics=True

)

In [ ]:
a_ift = compute_inverse_fourier_transform(a_ft)

In [ ]:
print(

    f'\nInverse Fourier transform output: \n{a_ift}'

)

In [ ]:
a_ifft = compute_inverse_fast_fourier_transform(a_ft)

In [ ]:
print(

    f'\nInverse fast Fourier transform output: \n{a_ifft}'

)

## Comparing inverse Fourier transform with two dimensional inverse fast Fourier transform:

In [ ]:
assert(

    np.isclose(

        a_ift, a_ifft, rtol=6

    )

).all()

## **Error in estimating the original input matrix using inverse discrete Fourier transform:**

In [ ]:
print(

    f'\nError between original 2D matrix input and inverse Fourier transform matrix: \n{np.expand_dims(a_ift.flatten(),-1) - np.expand_dims(a.flatten(),-1)}'

)

# Separable functions of discrete Fourier transform:

$
\text{The discrete Fourier transform function: }
$

$
{F(a,b)={{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}f(k,l){e^{{-i}{2}{\pi}{({\frac{ka}{N}}+{\frac{lb}{M}})}}}}
\text{, can be expressed as two separable functions:}$

${P(a,l)={{\sum\limits_{k=0}^{N-1}}}f(k,l){e^{{-i}{2}{\pi}{({\frac{ka}{N}})}}}}$

${F(a,b)={{\sum\limits_{l=0}^{M-1}}}P(a,l){e^{{-i}{2}{\pi}{({\frac{lb}{M}})}}}}$

In [ ]:
def separable_step_p_transform(

    inp_arr:np.ndarray,

    i:int, l:int,

    pt_coef:'Vector',

    transform_fn:callable,

    p_t_dict:dict=None,

    dtype:str='complex64',

    return_dict:bool=False

)->'Returns either a tuple or a vector, depending on the keyword argument: return_dict':

    N = inp_arr.shape[0]

    p_t_sum = 0

    assert callable(transform_fn), f'Expected a callable object or a function as the input for the keyword argument: transform_fn. instead received an instance of: {type(transform_fn)}'

    if p_t_dict is not None:

        assert isinstance(p_t_dict, dict), f'Expected a dictionary as the input for the keyword argument: p_ft_dict, instead received an instance of: {type(p_t_fict)} ...'

    else:

        p_t_dict = dict()

    for k in range(N):

        d = ((k * i) / N)

        d = d - math.floor(d)

        p_t, p_t_dict = lookup_and_update_transform_function_dict(

            d, p_t_dict, transform_fn, dtype

        )

        p_t_sum += inp_arr[k, l] * p_t * pt_coef

    if return_dict:

        return p_t_sum, p_t_dict

    return p_t_sum

In [ ]:
def separable_steps_transform(

    inp_arr:np.ndarray,

    i:int, j:int,

    transform_fn:callable,

    t_dict:dict=None,

    dtype:str='complex64',

    return_dict:bool=False

)->'Returns either a tuple or a vector, depending on the keyword argument: return_dict':

    M = inp_arr.shape[1]

    t_sum = 0

    assert callable(transform_fn), f'Expected a callable object or a function as the input for the keyword argument: transform_fn. instead received an instance of: {type(transform_fn)}'

    if t_dict is not None:

        assert isinstance(t_dict, dict), f'Expected a dictionary as the input for the keyword argument: p_ft_dict, instead received an instance of: {type(t_fict)} ...'

    else:

        t_dict = dict()

    for l in range(M):

        d = ((l * j) / M)

        d = d - math.floor(d)

        t_val, t_dict = lookup_and_update_transform_function_dict(

            d, t_dict, transform_fn, dtype

        )

        t_val, t_dict = separable_step_p_transform(

            inp_arr, i, l, t_val, transform_fn, t_dict, dtype, True

        )

        t_sum += t_val

    if return_dict:

        return t_sum, t_dict

    return t_sum


In [ ]:
def compute_separable_steps_transform(

    inp_arr:np.ndarray,

    transform_fn:callable,

    dtype:str='complex64',

    verbose:bool=False

)->np.ndarray:

    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'

    arr_t_sep = np.zeros_like(inp_arr).astype(getattr(np, dtype))

    N, M = arr_t_sep.shape[0], arr_t_sep.shape[1]

    if verbose:

        t_start_time = time.time()

    t_dict = dict()

    for i in range(N):

        for j in range(M):

            if verbose:

                start_time = time.time()

            t_val, t_dict = separable_steps_transform(

                inp_arr, i, j, transform_fn, t_dict, dtype, True

            )

            arr_t_sep[i, j] = t_val[0]

            if verbose:

                end_time = time.time()

                prog_frac = ((i * M) + j + 1) / (N * M)

                exe_time = end_time - start_time

                time_remaining = (1 - prog_frac) * exe_time * N * M

                print(

                        'Time remaining : ', time_remaining, ' (s)',

                        ' ' * 25,

                        end = '\r',

                        flush=True

                )

    if verbose:

        print(

                'Finished computing transformed matrix in: ', end_time - t_start_time, ' (s) ...',

                ' ' * 25,

                end = '\r',

                flush=True

        )

    return arr_t_sep

In [ ]:
def compute_fourier_separable_steps_transform(

    inp_arr:np.ndarray,

    dtype:str='complex64',

    verbose:bool=False

)->np.ndarray:

    return compute_separable_steps_transform(

        inp_arr, fourier_transform_exp, dtype, verbose

    )

In [ ]:
a_ft_sep = compute_fourier_separable_steps_transform(

    a, verbose=True

)

In [ ]:
print(

    f'\nOutput of the Fourier transform using separable functions: \n{a_ft_sep}'

)

## **Difference in results using different methods of evaluating discrete Fourier transform:**

In [ ]:
a_ft - a_ft_sep

In [ ]:
assert(

    np.isclose(

        compute_inverse_fourier_transform(a_fft),

        compute_inverse_fourier_transform(a_ft),

        rtol=6

    )

).all()

In [ ]:
print(

    np.fft.fft(a)[0][0] / np.sum(a)

)

print(

    np.isclose(np.fft.fft(a), a_ft)

)

# Separable functions of inverse discrete Fourier transform:

$
\text{The inverse discrete Fourier transform function: }
$

$
{f(x,y)={{\frac{1}{N{\cdot}M}}{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}F(k,l){e^{{i}{2}{\pi}{({\frac{kx}{N}}+{\frac{ly}{M}})}}}}
\text{, can be expressed as two separable functions:}$

${p(x,l)={{\frac{1}{N}}{\sum\limits_{k=0}^{N-1}}}F(k,l){e^{{i}{2}{\pi}{({\frac{kx}{N}})}}}}$

${f(x,y)={{\frac{1}{M}}{{\sum\limits_{l=0}^{M-1}}}}p(x,l){e^{{i}{2}{\pi}{({\frac{ly}{M}})}}}}$

In [ ]:
def compute_inverse_fourier_separable_steps_transform(

    inp_arr:np.ndarray,

    dtype:str='complex64',

    verbose:bool=False

)->np.ndarray:

    return compute_separable_steps_transform(

        inp_arr,

        inverse_fourier_transform_exp,

        dtype,

        verbose

    ) / np.prod(np.shape(inp_arr))

In [ ]:
a_ift_sep = compute_inverse_fourier_separable_steps_transform(a_ft_sep)

In [ ]:
print(

    f'\nOutput of the inverse Fourier transform using separable functions: \n{a_ift_sep}'

)

## **Error in estimating the original input matrix using inverse discrete Fourier transform:**

In [ ]:
assert(

    np.isclose(

        a_ift_sep, a, rtol=6

    )

).all()

In [ ]:
a_ift_sep - a

# Fourier transform of a discrete signal using NumPy:

In [ ]:
# Source - https://stackoverflow.com/q/52350774
# Posted by Luke Polson
# Retrieved 2026-04-03, License - CC BY-SA 4.0

# Sample spacing
dx = 1.0 / 1000.0

# Points
x1 = -5
x2 = 5

x = np.arange(x1, x2, dx)

def light_intensity():
    return 10 * sp.stats.norm.pdf(x, 0, 1)+ 0.1 * np.random.randn(x.size)

fig, ax = plt.subplots()
ax.plot(x,light_intensity())

In [ ]:
# Source - https://stackoverflow.com/a/52351218
# Posted by Mateen Ulhaq, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-03, License - CC BY-SA 4.0
# Modified by Dr. Rahul Remanan after retrieval

def norm_fft(y, T, max_freq=None):
    N = y.shape[0]
    Nf = N // 2 if max_freq is None else int(max_freq * T)
    xf = np.linspace(0.0, 0.5 * N / T, N // 2)
    yf = 2.0 / N * np.fft.fft(y)
    return xf[:Nf], yf[:Nf]

def generate_signal(x, signal_gain=10.0, noise_gain=0.0):
    signal = norm.pdf(x, 0, 1)
    noise = np.random.randn(x.size)
    return signal_gain * signal + noise_gain * noise

# Signal parameters
x1 = 0.0
x2 = 5.0
N = 10000
T = x2 - x1

# Generate signal data
x = np.linspace(x1, x2, N)
y = generate_signal(x)

# Apply FFT
xf, yf = norm_fft(y, T, T / np.pi)

# Plot
fig, ax = plt.subplots(2)
ax[0].plot(x, y)
ax[1].plot(xf, np.abs(yf))
plt.show()

In [ ]:
# Source - https://stackoverflow.com/a/52351218
# Posted by Mateen Ulhaq, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-03, License - CC BY-SA 4.0
# Modified by Dr. Rahul Remanan after retrieval

def norm_sym_fft(y, T, max_freq=None):
    N = y.shape[0]
    b = N if max_freq is None else int(max_freq * T + N // 2)
    a = N - b
    xf = np.linspace(-0.5 * N / T, 0.5 * N / T, N)
    yf = 2.0 / N * np.fft.fftshift(np.fft.fft(y))
    return xf[a:b], yf[a:b]

def generate_signal(x, signal_gain=10.0, noise_gain=0.0):
    signal = norm.pdf(x, 0, 1)
    noise = np.random.randn(x.size)
    return signal_gain * signal + noise_gain * noise

# Signal parameters
x1 = -10.0
x2 = 10.0
N = 10000
T = x2 - x1

# Generate signal data
x = np.linspace(x1, x2, N)
y = generate_signal(x)

# Apply FFT
xf, yf = norm_sym_fft(y, T, 4 / np.pi)

# Plot
fig, ax = plt.subplots(2)
ax[0].plot(x, y)
ax[1].plot(xf, np.abs(yf))
plt.show()

In [ ]:
# Source - https://stackoverflow.com/a/52351212
# Posted by Mahdi
# Retrieved 2026-04-03, License - CC BY-SA 4.0
# Modified by Dr. Rahul Remanan after retrieval

# Sample spacing
dx = 1.0 / 1000.0

# Points
x1 = -5
x2 = 5

x = np.arange(x1, x2, dx)

xf = np.arange(x1, x2, dx)
yf = np.fft.fft(light_intensity())
yfft = np.fft.fftshift(np.abs(yf))
fig,ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].plot(xf, light_intensity())
ax[1].plot(xf, yfft)
ax[1].set_xlim(-0.05, 0.05)
plt.show()

# [Fast Fourier transform of a grey scale image](https://homepages.inf.ed.ac.uk/rbf/HIPR2/fourier.htm):

In [ ]:
img_file = '../media/cln1.gif'

if not os.path.exists(img_file):

    !curl https://homepages.inf.ed.ac.uk/rbf/HIPR2/images/cln1.gif >> {img_file}

## **Fast Fourier transform:**

In [ ]:
img = mpimg.imread(img_file)

In [ ]:
imgplot = plt.imshow(

    np.expand_dims(

        img[ : , : , 0], axis=-1

    )

)

In [ ]:
img_fft = np.fft.fft(

    img[ : , : , 0]

)

## Two dimensional fast Fourier transform:

In [ ]:
img_two_dimensional_fft = compute_fast_fourier_transform(

    img[:, :, 0],

    verbose=True

)

In [ ]:
def min_max_scaler(img:np.ndarray)->np.ndarray:

    if np.max(img) - np.min(img) != 0:

        return (img - np.min(img)) / (np.max(img) - np.min(img))

    return img

In [ ]:
def log_transform(img:np.ndarray)->np.ndarray:

    return (1 / (np.log(1 + np.max(img)))) * np.log(1 + img)

### Plotting the implementation differences of fast Fourier transform:

#### Plot of the fast Fourier transform:

In [ ]:
img_fft_phase_plot = plt.imshow(

    np.expand_dims(

        log_transform(

            min_max_scaler(

                np.angle(

                    img_fft

                )

            )

        ), axis=-1

    )

)

#### Plot of the two dimensional fast Fourier transform:

In [ ]:
img_fft_mag_plot = plt.imshow(

    np.expand_dims(

        log_transform(

            (

                np.abs(

                    img_two_dimensional_fft

                )

            )

        ), axis=-1

    )

)

## **Inverse fast Fourier transform:**

In [ ]:
img_ifft = np.fft.ifft(

    img_fft

)

In [ ]:
img_fft_mag_plot = plt.imshow(

    np.abs(

        np.expand_dims(

            img_ifft,

            axis=-1

        )

    )

)

## Two dimensional inverse fast Fourier transform:

In [ ]:
img_two_dimensional_ifft = compute_inverse_fast_fourier_transform(

    img_two_dimensional_fft,

    verbose=True

)

In [ ]:
img_two_dimensional_fft_mag_plot = plt.imshow(

    np.abs(

        np.expand_dims(

            img_two_dimensional_ifft,

            axis=-1

        )

    )

)

## Inverse Fourier transform without the phase information:

In [ ]:
img_ifft_mag_plot = plt.imshow(

    np.expand_dims(

        log_transform(

            np.abs(

                np.fft.ifft(

                    np.abs(

                        img_fft

                    )

                )

            )

        ), axis=-1

    )

)

## Two dimensional inverse fast Fourier transform without the phase information:

In [ ]:
img_ifft_mag_plot = plt.imshow(

    np.expand_dims(

        log_transform(

            np.abs(

                np.fft.ifft(

                    np.abs(

                        img_two_dimensional_fft

                    )

                )

            )

        ), axis=-1

    )

)

## Inverse Fourier transform using magnitude and phase as separable components:

In [ ]:
img_ifft_mag_plot = plt.imshow(

    np.expand_dims(

        min_max_scaler(

            np.abs(

                np.fft.ifft(

                    np.abs(

                        img_fft

                    ) * np.sign(

                            img_fft

                        )

                )

            )

        ), axis=-1

    )

)

## Two dimensional inverse Fourier transform using separate magnitude and phase components:

In [ ]:
img_ifft_mag_plot = plt.imshow(

    np.expand_dims(

        min_max_scaler(

            np.abs(

                compute_inverse_fast_fourier_transform(

                    np.abs(

                        img_two_dimensional_fft

                    ) * np.sign(

                            img_two_dimensional_fft

                        ), 

                    verbose=True

                )

            )

        ), 

        axis=-1

    )

)